# NumCompute Quickstart Demo

This notebook demonstrates the final NumCompute package after the HD-level updates.

It covers:
- CSV loading
- preprocessing
- sorting and searching
- ranking and percentiles
- descriptive and streaming statistics
- classification and regression metrics
- ROC curve and AUC
- optimisation
- pipelines
- utilities
- benchmarking

## 1. Imports and project path setup

In [ ]:
import sys
from pathlib import Path

# Make imports work whether this notebook is run from the project root or demo/ folder.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "demo":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import tempfile

import numcompute as nc

from numcompute.io import load_csv
from numcompute.preprocessing import StandardScaler, MinMaxScaler, SimpleImputer, OneHotEncoder
from numcompute.sort_search import sort_array, lexsort_rows, multi_key_sort, topk, top_k, quickselect, binary_search
from numcompute.rank import rank, percentile
from numcompute.stats import (
    mean,
    median,
    std,
    min as stats_min,
    max as stats_max,
    histogram,
    quantile,
    StreamingStats,
)
from numcompute.metrics import (
    accuracy,
    precision,
    recall,
    f1,
    confusion_matrix,
    mse,
    roc_curve,
    auc,
)
from numcompute.optim import grad, jacobian
from numcompute.pipeline import Pipeline
from numcompute.utils import (
    check_array,
    sigmoid,
    relu,
    logsumexp,
    stable_softmax,
    euclidean_distance,
    manhattan_distance,
    cosine_similarity,
    pairwise_euclidean,
    topk_indices,
    topk_values,
    make_batches,
)
from numcompute.benchmarking import (
    compare_functions,
    vectorized_sum_of_squares,
    loop_sum_of_squares,
    format_benchmark_table,
)

print("NumCompute version:", nc.__version__)
print("Public exports available:", len(nc.__all__))

## 2. CSV loading with missing values

In [ ]:
csv_text = "1,10,0\n2,,0\n3,30,1\n4,40,1"

with tempfile.NamedTemporaryFile(mode="w+", suffix=".csv", delete=False) as f:
    f.write(csv_text)
    temp_path = f.name

data = load_csv(temp_path, delimiter=",")

print("Loaded CSV data:")
print(data)

## 3. Preprocessing: imputation, scaling, and one-hot encoding

In [ ]:
X_num = data[:, :2]
y = data[:, 2].astype(int)

imputer = SimpleImputer(strategy="mean", fill_value=0)
X_imputed = imputer.fit_transform(X_num)

standard_scaler = StandardScaler()
X_standard = standard_scaler.fit_transform(X_imputed)

minmax_scaler = MinMaxScaler(feature_range=(-1, 1))
X_minmax = minmax_scaler.fit_transform(X_imputed)

print("Original numeric data:")
print(X_num)

print("\nImputed data:")
print(X_imputed)

print("\nStandard scaled data:")
print(X_standard)

print("\nMinMax scaled data:")
print(X_minmax)

In [ ]:
X_cat = np.array([
    ["red", "small"],
    ["blue", "large"],
    ["red", "large"],
    ["green", "small"],
], dtype=object)

encoder = OneHotEncoder(handle_unknown="ignore")
X_encoded = encoder.fit_transform(X_cat)

print("Categories learned:")
print(encoder.categories_)

print("\nOne-hot encoded data:")
print(X_encoded)

print("\nUnknown category with handle_unknown='ignore':")
print(encoder.transform(np.array([["purple", "small"]], dtype=object)))

## 4. Sorting, searching, top-k, and quickselect

In [ ]:
values = np.array([10, 30, 20, 20, 50, 40])

print("Original values:", values)
print("Stable sort:", sort_array(values))

print("\nTop-3 largest indices using topk:")
print(topk(values, k=3, largest=True, return_indices=True))

print("\nTop-3 largest values using topk:")
print(topk(values, k=3, largest=True, return_indices=False))

print("\nBackward-compatible top_k values:")
print(top_k(values, k=3, largest=True))

print("\nQuickselect k=2 (third smallest value):")
print(quickselect(values, 2))

sorted_values = sort_array(values)
print("\nBinary search for 30:")
print(binary_search(sorted_values, 30))

print("\nBinary search for missing value 25:")
print(binary_search(sorted_values, 25))

In [ ]:
rows = np.array([
    [2, 1],
    [1, 2],
    [1, 1],
    [2, 0],
])

print("Rows:")
print(rows)

print("\nLexsort rows by columns [0, 1]:")
print(lexsort_rows(rows, keys=[0, 1]))

print("\nMulti-key sort alias:")
print(multi_key_sort(rows, keys=[0, 1]))

## 5. Ranking and percentiles

In [ ]:
rank_values = np.array([10, 20, 20, 30, np.nan])

print("Values:", rank_values)
print("Average ranks:", rank(rank_values, method="average"))
print("Dense ranks:", rank(rank_values, method="dense"))
print("Ordinal ranks:", rank(rank_values, method="ordinal"))

clean_values = np.array([1, 2, 3, 4])

print("\nPercentile 50 linear:", percentile(clean_values, 50, interpolation="linear"))
print("Percentile 50 lower:", percentile(clean_values, 50, interpolation="lower"))
print("Percentile 50 higher:", percentile(clean_values, 50, interpolation="higher"))
print("Percentile 50 midpoint:", percentile(clean_values, 50, interpolation="midpoint"))

## 6. Descriptive statistics and StreamingStats

In [ ]:
stats_values = np.array([1, 2, np.nan, 3, 4, 5], dtype=float)

print("Mean:", mean(stats_values))
print("Median:", median(stats_values))
print("Standard deviation:", std(stats_values))
print("Min:", stats_min(stats_values))
print("Max:", stats_max(stats_values))
print("Quantile 0.5:", quantile(stats_values, 0.5))

counts, bins = histogram(stats_values, bins=3)
print("\nHistogram counts:", counts)
print("Histogram bins:", bins)

matrix = np.array([[1, 2, np.nan], [4, 5, 6]], dtype=float)
print("\nAxis-wise mean over rows:")
print(mean(matrix, axis=1))

In [ ]:
stream = StreamingStats()
stream.update_many([1, 2, 3, np.nan, 4, 5])

print("Streaming count:", stream.count)
print("Streaming mean:", stream.mean)
print("Streaming variance:", stream.variance)
print("Streaming std:", stream.std)
print("Streaming min:", stream.min)
print("Streaming max:", stream.max)

print("\nStreaming summary dictionary:")
print(stream.to_dict())

## 7. Metrics: classification, regression, ROC, and AUC

In [ ]:
y_true = np.array([0, 1, 1, 0, 1])
y_pred = np.array([0, 1, 0, 0, 1])

print("Accuracy:", accuracy(y_true, y_pred))
print("Precision:", precision(y_true, y_pred))
print("Recall:", recall(y_true, y_pred))
print("F1:", f1(y_true, y_pred))
print("Confusion matrix:")
print(confusion_matrix(y_true, y_pred))

y_true_reg = np.array([1.0, 2.0, 3.0])
y_pred_reg = np.array([1.2, 1.9, 2.8])

print("\nMSE:", mse(y_true_reg, y_pred_reg))

In [ ]:
y_true_roc = np.array([0, 0, 1, 1])
y_scores = np.array([0.1, 0.4, 0.35, 0.8])

fpr, tpr, thresholds = roc_curve(y_true_roc, y_scores)
roc_auc = auc(fpr, tpr)

print("FPR:", fpr)
print("TPR:", tpr)
print("Thresholds:", thresholds)
print("AUC:", roc_auc)

## 8. Optimisation: gradient and Jacobian

In [ ]:
def f_scalar(x):
    return x[0] ** 2 + x[1] ** 2

def f_vector(x):
    return np.array([
        x[0] + x[1],
        x[0] * x[1],
    ])

x = np.array([3.0, 4.0])

print("Gradient:")
print(grad(f_scalar, x))

print("\nJacobian:")
print(jacobian(f_vector, x))

## 9. Pipeline

In [ ]:
class DummyModel:
    def fit(self, X, y=None):
        self.threshold_ = np.mean(X[:, 0])
        return self

    def predict(self, X):
        return (X[:, 0] > self.threshold_).astype(int)

pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()),
    ("model", DummyModel()),
])

pipe.fit(X_num, y)
preds = pipe.predict(X_num)

print("Pipeline predictions:")
print(preds)

## 10. Utility functions

In [ ]:
a = np.array([0, 0])
b = np.array([3, 4])

print("Euclidean distance:", euclidean_distance(a, b))
print("Manhattan distance:", manhattan_distance(a, b))
print("Cosine similarity:", cosine_similarity([1, 0], [1, 0]))

scores = np.array([1.0, 2.0, 3.0])
print("\nSigmoid:", sigmoid(scores))
print("ReLU:", relu(np.array([-1.0, 0.0, 2.0])))
print("LogSumExp:", logsumexp(scores))
print("Stable softmax:", stable_softmax(scores))

X = np.array([[0.0, 0.0], [1.0, 0.0]])
Y = np.array([[0.0, 1.0]])

print("\nPairwise Euclidean:")
print(pairwise_euclidean(X, Y))

values_for_topk = np.array([1, 9, 3, 7])
print("\nTop-k indices:", topk_indices(values_for_topk, 2))
print("Top-k values:", topk_values(values_for_topk, 2))

print("\nBatches:")
for batch in make_batches(np.arange(10), batch_size=4):
    print(batch)

## 11. Benchmarking

In [ ]:
benchmark_values = np.random.default_rng(42).random(100_000)

result = compare_functions(
    vectorized_sum_of_squares,
    loop_sum_of_squares,
    benchmark_values,
    repeat=5,
    warmup=1,
)

print(format_benchmark_table([result["first"], result["second"]]))
print("Speedup:", f"{result['speedup_vs_slow']:.2f}x")

## Summary

This final quickstart notebook demonstrates the integrated package end-to-end.

It includes the latest updates to:
- preprocessing
- sorting/searching
- ranking
- statistics and streaming statistics
- metrics including ROC/AUC
- utilities
- package-level imports
- benchmarking